In [1]:
import pandas as pd
import numpy as np

from lightgbm import LGBMRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [2]:
data = pd.read_csv('Data/Final_Data.csv')

In [3]:
data = data.drop(columns=["SiteKey"])
data.shape

(1599443, 36)

In [4]:
data.columns

Index(['SolarGeneration', 'ApparentTemperature', 'AirTemperature',
       'DewPointTemperature', 'RelativeHumidity', 'WindSpeed',
       'CapacityFactor', 'WindDir_sin', 'WindDir_cos', 'Hour_sin', 'Hour_cos',
       'Day_sin', 'Day_cos', 'SolarPotential', 'Temp_solar', 'Humidity_solar',
       'Wind_solar', 'Temp_change', 'Humidity_change', 'Wind_change', 'Lag_1',
       'Lag_2', 'Lag_3', 'Ramp_1', 'Ramp_2', 'RollingStd_4', 'CF_smooth_4',
       'CloudinessIndex', 'CloudTrend', 'Target_15min', 'Target_1h',
       'Target_3h', 'Residual_now', 'Residual_15min', 'Residual_1h',
       'Residual_3h'],
      dtype='object')

In [5]:
X_now = data[[
    "Day_sin","Day_cos",
    "Hour_sin","Hour_cos",
    "Lag_1","Lag_2","Lag_3",
    "Ramp_1","Ramp_2",
    "RollingStd_4","CF_smooth_4","SolarPotential"
]]

y_now = data["Residual_now"]
true_now = data["CapacityFactor"]

n_splits = 5
split_size = int(len(X_now) / (n_splits + 1))
max_train_size = 1599443

mae_model_now = 0
mae_naive_now = 0
r2_now = 0

for i in range(n_splits):
    train_end = split_size * (i+1)
    test_end = split_size * (i+2)
    train_start = max(0, train_end - max_train_size)

    X_train = X_now.iloc[train_start:train_end]
    X_test = X_now.iloc[train_end:test_end]

    y_train = y_now.iloc[train_start:train_end]
    true_test = true_now.iloc[train_end:test_end]

    lag_test = X_test["Lag_1"].values

    model = LGBMRegressor(
        n_estimators=400,
        learning_rate=0.1,
        max_depth=18,
        num_leaves=256,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=20,
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
    )

    model.fit(X_train, y_train)
    residual_pred = model.predict(X_test)

    pred = lag_test + residual_pred

    mae_model_now += mean_absolute_error(true_test, pred)
    mae_naive_now += mean_absolute_error(true_test, lag_test)
    r2_now += r2_score(true_test, pred)

mae_model_now /= n_splits
mae_naive_now /= n_splits
r2_now /= n_splits

print("\n=== MEMORY MODEL: CURRENT ===")
print("Naive MAE:", mae_naive_now)
print("Model MAE:", mae_model_now)
print("R2:", r2_now)


=== MEMORY MODEL: CURRENT ===
Naive MAE: 0.00891871835001148
Model MAE: 0.006877399556858498
R2: 0.9525137588062869


In [6]:
importance_now = pd.DataFrame({
    "Feature": X_now.columns,
    "Importance": model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("\n=== CURRENT MODEL FEATURE IMPORTANCE ===")
print(importance_now.head(20))


=== CURRENT MODEL FEATURE IMPORTANCE ===
           Feature  Importance
11  SolarPotential       17237
0          Day_sin       14625
1          Day_cos       10222
7           Ramp_1        9488
8           Ramp_2        9310
9     RollingStd_4        9236
4            Lag_1        7493
6            Lag_3        6470
5            Lag_2        5832
10     CF_smooth_4        5485
2         Hour_sin        4383
3         Hour_cos        2216


In [7]:
X = data[[
    "Day_sin","Day_cos",
    "Hour_sin","Hour_cos",
    "Lag_1","Lag_2","Lag_3",
    "Ramp_1","Ramp_2",
    "RollingStd_4","CF_smooth_4","SolarPotential",
    "Residual_now"
]]
Y = data[["Residual_15min", "Residual_1h", "Residual_3h"]]
Y_true = data[["Target_15min", "Target_1h", "Target_3h"]]

n_splits = 5
split_size = int(len(X) / (n_splits + 1))
max_train_size = 1599443

mae_model = np.zeros(3)
mae_naive = np.zeros(3)
r2_scores = np.zeros(3)

for i in range(n_splits):
    train_end = split_size * (i+1)
    test_end = split_size * (i+2)
    train_start = max(0, train_end - max_train_size)

    X_train = X.iloc[train_start:train_end]
    X_test = X.iloc[train_end:test_end]

    Y_train = Y.iloc[train_start:train_end]
    true_test = Y_true.iloc[train_end:test_end]

    lag_test_15 = X_test["Lag_1"].values
    lag_test_1h = X_test["Lag_2"].values
    lag_test_3h = X_test["Lag_3"].values
    lag_tests = [lag_test_15, lag_test_1h, lag_test_3h]

    model = MultiOutputRegressor(
        LGBMRegressor(
            n_estimators=400,
            learning_rate=0.1,
            max_depth=18,
            num_leaves=256,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_samples=20,
            random_state=42,
            n_jobs=-1,
            verbosity=-1,
        )
    )

    model.fit(X_train, Y_train)
    residual_pred = model.predict(X_test)

    pred = np.zeros_like(residual_pred)

    for h in range(3):
        pred[:, h] = lag_tests[h] + residual_pred[:, h]

    for h in range(3):
        mae_model[h] += mean_absolute_error(true_test.iloc[:, h], pred[:, h])
        mae_naive[h] += mean_absolute_error(true_test.iloc[:, h], lag_tests[h])
        r2_scores[h] += r2_score(true_test.iloc[:, h], pred[:, h])

mae_model /= n_splits
mae_naive /= n_splits
r2_scores /= n_splits

horizons = ["15 min","1 hour","3 hour"]

print("\n=== MEMORY MODEL: FUTURE ===")
for i, h in enumerate(horizons):
    print(f"\n{h}")
    print("Naive MAE:", mae_naive[i])
    print("Model MAE:", mae_model[i])
    print("R2:", r2_scores[i])


=== MEMORY MODEL: FUTURE ===

15 min
Naive MAE: 0.013283456971665514
Model MAE: 0.007397974016633091
R2: 0.9494720184979331

1 hour
Naive MAE: 0.025374984928098875
Model MAE: 0.016332869865210823
R2: 0.8157065272865452

3 hour
Naive MAE: 0.04857314903658821
Model MAE: 0.025921857297942393
R2: 0.6125749130211664


In [9]:
horizons = ["15 min","1 hour","3 hour"]

for i, h in enumerate(horizons):
    lgb_model = model.estimators_[i]
    importance = lgb_model.booster_.feature_importance(importance_type='gain')
    feat_imp = pd.DataFrame({
        "Feature": X.columns,
        "Importance": importance
    }).sort_values(by="Importance", ascending=False)
    
    print(f"\n=== TOP FEATURES: {h} ===")
    print(feat_imp.head(30))


=== TOP FEATURES: 15 min ===
           Feature   Importance
12    Residual_now  3793.415992
11  SolarPotential   664.560300
4            Lag_1   511.800637
0          Day_sin   357.480195
7           Ramp_1   310.408788
1          Day_cos   298.755896
2         Hour_sin   250.589836
8           Ramp_2   245.893388
9     RollingStd_4   218.697690
6            Lag_3   215.687881
10     CF_smooth_4   174.544698
5            Lag_2   142.591780
3         Hour_cos    93.976549

=== TOP FEATURES: 1 hour ===
           Feature   Importance
12    Residual_now  4418.549449
4            Lag_1  2341.547570
11  SolarPotential  1907.186562
2         Hour_sin  1875.511381
10     CF_smooth_4  1590.067811
0          Day_sin   877.963993
1          Day_cos   706.239702
7           Ramp_1   607.945739
3         Hour_cos   564.311069
9     RollingStd_4   554.214870
8           Ramp_2   437.061238
6            Lag_3   389.889188
5            Lag_2   321.372053

=== TOP FEATURES: 3 hour ===
           Fea